In [1]:
file = "russel1000_News.csv"

stock_symbol = "rui" #e.g. tsla, msft
column = 'Summary' # which column to clean

output_filename = "russel1000_News_cleaned.csv"

# Combine csv files (if needed)

'''ensure that all files csv files you want to combine are in the same folder '''

In [2]:
import os
import glob
import pandas as pd

path = 'D:/Programming/Jupyter/IS453 - Financial Analytics/Github/Data/News/russel1000_News_google'
extension = 'csv'
os.chdir(path)
output_filename = path.split('/')[-1] + '_combined.csv'

all_files = [i for i in glob.glob('*.{}'.format(extension))]

combined_csv = pd.concat([pd.read_csv(f) for f in all_files])
combined_csv.to_csv(output_filename, index = False, encoding = 'utf-8-sig')

# Cleaning text

In [3]:
import re
import nltk
import numpy as np
import pandas as pd
from nltk import word_tokenize
from nltk.tokenize import RegexpTokenizer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

In [4]:
# text cleaning
def text_lower(text):
    return ' '.join([t.lower() for t in text.split()])

def remove_hashtag_mentions_urls(text):
    return re.sub(r"(?:\@|\#|https?\://)\S+", "", text)

def remove_emoji(text):
    emoji_pattern = re.compile("["
    u"\U0001F600-\U0001F64F"  # emoticons
    u"\U0001F300-\U0001F5FF"  # symbols & pictographs
    u"\U0001F680-\U0001F6FF"  # transport & map symbols
    u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
    u"\U00002702-\U000027B0"
    u"\U000024C2-\U0001F251"
    "]+", flags=re.UNICODE)

    return emoji_pattern.sub(r'', text)

def tokenization(text):
#     word_tokenizer = RegexpTokenizer(r'[-\'\w]+')
    word_tokenizer = RegexpTokenizer(r'\w+')
    tokenized_text = word_tokenizer.tokenize(text)
    return tokenized_text

def remove_numbers(text):
    return [t for t in text if not t.isdigit()]

def remove_stopwords(text, symbol):
    word_dictionary = {
        'aapl': ['$AAPL', 'apple'],
        'googl': ['$GOOGL', 'google'],
        'msft': ['$MSFT', 'microsoft'],
        'tsla': ['$TSLA', 'tesla'],
        'amzn': ['$AMZN', 'amazon'],
        'vxrt': ['$VXRT', 'vaxart'],
        'rui': ['$RUI', 'russell', 'index'],
        'ark': ['$ARK', 'resources'],
        'ater': ['$ATER', 'Aterian']
    }
    stop_words = stopwords.words('english')
    stop_words.append(symbol)
    
    try:
        for word in word_dictionary[symbol]:
            stop_words.append(word)
    except:
        pass 
        
    text_stopremoved = [w for w in text if w not in stop_words]
    return text_stopremoved

def lemmatization(text):
    lemma_word = []
    wordnet_lemmatizer = WordNetLemmatizer()
    for w in text:
        word1 = wordnet_lemmatizer.lemmatize(w, pos = "n")
        word2 = wordnet_lemmatizer.lemmatize(word1, pos = 'v')
        word3 = wordnet_lemmatizer.lemmatize(word2, pos = ('a'))
        lemma_word.append(word3)
    return ' '.join(lemma_word)
    

def clean_text(text, symbol):
    text = text_lower(text)
    text = remove_hashtag_mentions_urls(text)
    text = remove_emoji(text)
    text = tokenization(text)
    text = remove_numbers(text)
    text = remove_stopwords(text, symbol)
    text = lemmatization(text)
    return text 

# Comment Classifier

In [5]:
def comment_classifier(file, column, stock_symbol): 
    df = pd.read_csv(file)
    
    #cleaning text
    text = df[column].to_list()
    cleaned_text = [clean_text(str(t), stock_symbol) if len(str(t)) > 0 else '' for t in text]

    output = 'cleaned ' + column
    df[output] = cleaned_text
    
    #remove missing values from updated df 
    df[output].replace('', np.nan, inplace = True)
    df.dropna(subset=[output], inplace = True)
    
    #for twitter
    try:
        #df = df[df[output].apply(lambda x: len(x)>4)]
        df = df[df['Number of Retweets'] > 5]
        df = df[df['Number of Likes'] > 200]
        df = df[df['Number of Followers'] > 50]
        df = df[df[output].str.split().str.len() > 4]

    except:
        pass
    
    
    return df

In [6]:
df = comment_classifier(file, column, stock_symbol)

df.head(3)

,Unnamed: 0,Ticker,Title,Article,Summary,Link,Date,cleaned Title,cleaned Summary
0,0,russel 1000,Apple's new executive bonus formula is designe...,In this article AAPL\r\n\r\nApple chief execut...,The new executive compensation measure reveale...,https://www.cnbc.com/2021/01/16/apple-ceo-tim-...,2021-01-16 00:00:00,apple new executive bonus formula design fast ...,new executive compensation measure reveal appl...
1,1,russel 1000,Looking Back at 2020: Great Year for Stocks Am...,The Santa Claus rally did not disappoint in 20...,The broadening of the equity market rally cont...,https://www.etftrends.com/etf-strategist-chann...,2021-01-07 16:20:20+00:00,look back great year stock amid tough year,broaden equity market rally continue december ...
2,2,russel 1000,Are you a robot?,Why did this happen?\r\n\r\nPlease make sure y...,Why did this happen?\r\nPlease make sure your ...,https://www.bloomberg.com/news/articles/2021-0...,2021-01-14 00:00:00,robot,happen please make sure browser support javasc...


In [7]:
df.to_csv(output_filename, index = False)